# ROC Curves, AUC & Threshold Optimization Lab

A single decision threshold provides only a snapshot of model behavior. The **Receiver Operating Characteristic (ROC)** curve evaluates classifier discrimination across every possible decision threshold from $0.0$ to $1.0$. This lab demonstrates how to plot ROC curves using scikit-learn, calculate the **Area Under the Curve (AUC)**, optimize decision thresholds using **Youden's J statistic**, and contrast ROC with **Precision-Recall (PR)** curves on severely imbalanced cohorts.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve, auc, roc_auc_score, precision_recall_curve,
    confusion_matrix, precision_score, recall_score
)
from sklearn.linear_model import LogisticRegression

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

## 1. Plotting the ROC Curve and Computing AUC

Fit a logistic regression model on a 100-sample dataset (70 negative, 30 positive) and extract the ROC trajectory.

In [ ]:
# Generate 2D synthetic features
n_neg, n_pos = 70, 30
X_neg = np.random.randn(n_neg, 2) - 1.2
X_pos = np.random.randn(n_pos, 2) + 1.2
X = np.vstack([X_neg, X_pos])
y = np.concatenate([np.zeros(n_neg), np.ones(n_pos)])

clf = LogisticRegression(random_state=42).fit(X, y)
y_scores = clf.predict_proba(X)[:, 1]

fpr, tpr, thresholds = roc_curve(y, y_scores)
roc_auc = auc(fpr, tpr)

print(f"ROC AUC Score: {roc_auc:.4f}")
print(f"\nSample Operating Points along the ROC Trajectory:")
print(f"{'Threshold':<12} {'FPR':<10} {'TPR (Recall)':<14} {'Youden J'}")
print("-" * 50)
for i in [1, len(thresholds)//4, len(thresholds)//2, 3*len(thresholds)//4, len(thresholds)-1]:
    j_val = tpr[i] - fpr[i]
    print(f"{thresholds[i]:<12.3f} {fpr[i]:<10.1%} {tpr[i]:<14.1%} {j_val:.3f}")

## 2. Threshold Tuning: Finding the Optimal Youden's Index

Find the operating point that maximizes **Youden's J statistic** ($J = \text{TPR} - \text{FPR} = \text{Sensitivity} + \text{Specificity} - 1$).

In [ ]:
youden_j = tpr - fpr
best_idx = np.argmax(youden_j)
best_thresh = thresholds[best_idx]

print(f"Optimal Threshold: {best_thresh:.3f}")
print(f"Maximum Youden J:  {youden_j[best_idx]:.3f}")
print(f"Sensitivity (TPR): {tpr[best_idx]:.1%}")
print(f"Specificity (TNR): {1.0 - fpr[best_idx]:.1%}")
print(f"False Alarm (FPR): {fpr[best_idx]:.1%}")

## 3. When ROC Lies: Comparing ROC-AUC vs. PR-AUC on Imbalanced Data

Simulate a severe 98:2 imbalanced dataset (980 negative, 20 positive) to demonstrate why PR curves are far more sensitive to minority performance.

In [ ]:
n_n, n_p = 980, 20
X_imb = np.vstack([np.random.randn(n_n, 2) - 0.4, np.random.randn(n_p, 2) + 0.8])
y_imb = np.concatenate([np.zeros(n_n), np.ones(n_p)])

clf_imb = LogisticRegression(random_state=42).fit(X_imb, y_imb)
scores_imb = clf_imb.predict_proba(X_imb)[:, 1]

roc_auc_val = roc_auc_score(y_imb, scores_imb)
p_vals, r_vals, _ = precision_recall_curve(y_imb, scores_imb)
pr_auc_val = auc(r_vals, p_vals)

print(f"Imbalanced Dataset (2.0% positive prevalence):")
print(f"  ROC-AUC Score: {roc_auc_val:.3f} (Looks deceptively high due to huge TN count)")
print(f"  PR-AUC Score:  {pr_auc_val:.3f} (Exposes the high volume of false alarms)")
print("\nTakeaway: On severe class imbalance, always benchmark PR-AUC alongside ROC-AUC.")